En este código se pone a prueba el modelo y el pipeline definido. Se usan además del modelo fine tuneado, dos modelos más, uno para diarizar (identificar hablantes) y otro para segmentar. Así podemos pasarle cada intervención a nuestro modelo de emociones y saber a quién le está hablando el interlocutor.

In [9]:
import torch
import torchaudio 
import numpy as np
import whisper
import spacy
import json
import subprocess
from pyannote.audio import Pipeline
from transformers import WavLMForSequenceClassification, Wav2Vec2FeatureExtractor
from pydub import AudioSegment
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("hf_token")

In [ ]:
# --- 0. CONFIGURACIÓN DE DISPOSITIVO (SOLUCIÓN AL ERROR) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Trabajando en: {device}")

# --- 1. CARGA DE MODELOS ---
try:
    nlp = spacy.load("en_core_web_sm") 
except:
    print("Descargando modelo de spaCy...")
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Configuración de Rutas
NOTEBOOK_DIR = os.getcwd()
MODEL_PATH = os.path.join(NOTEBOOK_DIR, "RAVDESS_WavLM_Final")
VIDEO_PATH = os.path.join(NOTEBOOK_DIR, "../solucion_1_audio_prompt/input/office.mp3") 

print("Cargando Modelos de IA...")

# Modelo de Emociones fine tuneado
emotion_model = WavLMForSequenceClassification.from_pretrained(
    MODEL_PATH, 
    use_safetensors=True
).to(device)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_PATH)
whisper_model = whisper.load_model("base")

# diarizacion para identificar hablantes
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", 
    token=HF_TOKEN
)
diarization_pipeline.to(device)

# --- 2. LÓGICA DE AUTO-IDENTIFICACIÓN DINÁMICA ---

def auto_identify_speakers(transcription_segments, diarization_result):
    speaker_guesses = {}
    
    for turn, _, speaker_id in diarization_result.itertracks(yield_label=True):
        text = ""
        for seg in transcription_segments:
            if (seg['start'] <= turn.end) and (seg['end'] >= turn.start):
                text += " " + seg['text']
        
        doc = nlp(text)
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                name = ent.text.strip().upper()
                if speaker_id not in speaker_guesses: speaker_guesses[speaker_id] = {}
                speaker_guesses[speaker_id][name] = speaker_guesses[speaker_id].get(name, 0) + 1

    final_mapping = {}
    used_names = set()
    
    for sid, names in speaker_guesses.items():
        if names:
            sorted_names = sorted(names.items(), key=lambda x: x[1], reverse=True)
            for name, freq in sorted_names:
                if name not in used_names:
                    final_mapping[sid] = name
                    used_names.add(name)
                    break
        if sid not in final_mapping:
            final_mapping[sid] = sid 

    return final_mapping

# --- 3. PROCESAMIENTO PRINCIPAL ---

def analyze_any_audio(file_path):
    audio_path = os.path.abspath("temp_audio.wav")
    file_path_abs = os.path.abspath(file_path)

    print(f"Convirtiendo {file_path_abs} a WAV...")
    
    command = [
        "ffmpeg", "-i", file_path_abs,
        "-ac", "1", "-ar", "16000",
        audio_path, "-y"
    ]
    
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"❌ Error en FFmpeg: {e.stderr}")

    if not os.path.exists(audio_path):
        raise FileNotFoundError(f"❌ FFmpeg no generó el archivo en: {audio_path}")

    # A. Diarización y Transcripción
    print("Cargando audio en memoria...")
    waveform, sr = torchaudio.load(audio_path)
    
    print("Identificando hablantes (Diarización)...")
    diarization = diarization_pipeline({"waveform": waveform, "sample_rate": sr})
    
    print("Transcribiendo audio (Whisper)...")
    transcription = whisper_model.transcribe(audio_path)

    # B. Auto-Identificar quién es quién
    print("Identificando personajes dinámicamente...")
    dynamic_mapping = auto_identify_speakers(transcription['segments'], diarization.speaker_diarization)

    # C. Análisis de Emociones y Menciones
    full_audio = AudioSegment.from_wav(audio_path)
    results = []

    print("Procesando segmentos emocionales...")
    for turn, _, speaker_id in diarization.speaker_diarization.itertracks(yield_label=True):
        if (turn.end - turn.start) < 0.5: continue

        speaker_name = dynamic_mapping.get(speaker_id, speaker_id)

        # Cortar audio
        seg_audio = full_audio[int(turn.start*1000):int(turn.end*1000)]
        samples = np.array(seg_audio.get_array_of_samples()).astype(np.float32) / 32768.0
        
        inputs = feature_extractor(
            samples, 
            sampling_rate=16000, 
            padding=True, 
            do_normalize=True, 
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            logits = emotion_model(**inputs).logits
            emotion = emotion_model.config.id2label[torch.argmax(logits, dim=-1).item()]

        # Texto y Menciones
        best_text = ""
        for seg in transcription['segments']:
            overlap = min(turn.end, seg['end']) - max(turn.start, seg['start'])
            if overlap > 0.1: # Al menos 0.1s de coincidencia
                best_text = seg['text'].strip()
                break

        doc = nlp(best_text)
        mentions = list(set([ent.text.strip().upper() for ent in doc.ents if ent.label_ == "PERSON"]))

        mentions = [m for m in mentions if m != speaker_name.upper()]


        results.append({
            "start": round(turn.start, 2),
            "end": round(turn.end, 2),
            "speaker": speaker_name,
            "emotion": emotion,
            "mentions": mentions,
            "text": best_text
        })

    # Guardar JSON
    output_json = "dynamic_scene_analysis.json"
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)
    
    try:
        if os.path.exists(audio_path):
            os.remove(audio_path)
    except:
        print(f"Aviso: No se pudo borrar {audio_path}, puede estar en uso.")

    return results

# --- EJECUCIÓN ---
try:
    results = analyze_any_audio(VIDEO_PATH)
    print(f"✅ ¡Éxito! Se ha generado '{os.getcwd()}/dynamic_scene_analysis.json'")
    for r in results[:5]:
        print(f"[{r['start']}s] {r['speaker']} ({r['emotion']}): {r['text'][:50]}...")
except Exception as e:
    print(f"❌ Error durante la ejecución: {e}")

Trabajando en: cpu
Cargando Modelos de IA...
Convirtiendo d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_1_audio_prompt\input\office.mp3 a WAV...
Cargando audio en memoria...
Identificando hablantes (Diarización)...


d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pyt

Transcribiendo audio (Whisper)...


d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Identificando personajes dinámicamente...
Procesando segmentos emocionales...


d:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning\venv\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


✅ ¡Éxito! Se ha generado 'd:\Users\pedro\Documents\GitHub\PGLN-Analisis-de-Sentimiento-Conversacional\solucion_2_audio_finetuning/dynamic_scene_analysis.json'
[0.15s] JIM (angry): How about a toast?...
[1.28s] JIM (surprise): Do you like it?...
[2.61s] JIM (disgust): Here's two good friends....
[5.06s] MICHAEL (disgust): Cheers....
[11.67s] JIM (disgust): That is sort of an oaky after birth....


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import imageio
import os
import pandas as pd

# 1. Mapa de colores
color_map = {
    'neutral': 'grey', 'happy': 'green', 'sad': 'blue',
    'angry': 'red', 'fear': 'purple', 'surprise': 'orange', 'disgust': 'brown'
}

def create_interaction_gif(results_list, output_gif="social_emotion_graph.gif", frame_duration=10.0):
    """
    frame_duration: Segundos que dura cada imagen. 
                    Aumenta este número para que el GIF vaya más lento (ej: 2.0 o 3.0).
    """
    df = pd.DataFrame(results_list)
    
    # 1. Nodos
    df['speaker'] = df['speaker'].str.upper()
    all_speakers = set(df['speaker'].unique())
    all_mentions = set([m.upper() for sublist in df['mentions'] for m in sublist])
    all_nodes = list(all_speakers.union(all_mentions))
    
    G = nx.DiGraph()
    G.add_nodes_from(all_nodes)
    pos = nx.circular_layout(G) 

    filenames = []
    last_speaker = None # Para saber quién habló antes

    for i, row in df.iterrows():
        plt.figure(figsize=(10, 8))
        current_speaker = row['speaker']
        emotion = row['emotion']
        mentions = [m.upper() for m in row['mentions']]
        current_color = color_map.get(emotion, 'black')

        # DIBUJAR NODOS
        node_colors = ['gold' if node == current_speaker else 'lightgray' for node in G.nodes]
        nx.draw_networkx_nodes(G, pos, node_size=3000, node_color=node_colors, edgecolors='black')
        nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

        # --- LÓGICA DE ARISTAS ---
        targets = []
        edge_label = ""

        if mentions:
            # PRIORIDAD 1: Menciones directas por nombre
            targets = [m for m in mentions if m in G.nodes]
            edge_label = emotion.upper()
        elif last_speaker and last_speaker != current_speaker:
            # PRIORIDAD 2: Si no hay nombre, reacciona al hablante anterior (Contexto)
            targets = [last_speaker]
            edge_label = f"({emotion.upper()})"

        if targets:
            edges_to_draw = [(current_speaker, t) for t in targets]
            nx.draw_networkx_edges(G, pos, edgelist=edges_to_draw, edge_color=current_color, 
                                   width=5, arrowsize=30, connectionstyle="arc3,rad=0.2")
            
            # Etiquetas sobre las aristas
            edge_labels = {edge: edge_label for edge in edges_to_draw}
            nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color=current_color, rotate=True)

        plt.title(f"TIEMPO: {row['start']}s\n{current_speaker}: {row['text'][:60]}...")
        plt.axis('off')
        
        filename = f"int_frame_{i:03d}.png"
        plt.savefig(filename, dpi=100)
        plt.close()
        filenames.append(filename)
        
        last_speaker = current_speaker # Actualizamos para el siguiente frame
        
    # --- COMPILAR EL GIF ---
    print(f"Compilando GIF: {output_gif}...")
    with imageio.get_writer(output_gif, mode='I', duration=frame_duration, loop=0) as writer:
        for filename in filenames:
            image = imageio.v3.imread(filename)
            writer.append_data(image)
            os.remove(filename)

    print(f"✅ GIF guardado como: {output_gif}")

results = json.load(open("dynamic_scene_analysis.json", "r", encoding="utf-8"))
create_interaction_gif(results, frame_duration=1000.0)

Compilando GIF: social_emotion_graph.gif...
✅ GIF guardado como: social_emotion_graph.gif


El código muestra buenos resultados e incluso detecta cambios en el propio tono de una misma intervención. Aunque esto sole es visible en el json, ya que para el gif creíamos que podia ser muy confuso.

El sistema es capaz de identificar correctamente a los hablantes, cada intervencion y las emociones expresadas entre cada uno de ellos.